## Крок 1. Опис фінального проєкту

Мета проєкту — створити AI-систему для збагачення Excel-даних. Система приймає Excel-файл або онлайн-посилання на Excel/Google Sheets та текстовий опис задачі, аналізує структуру таблиці, добуває потрібні дані з реальних веб/API-джерел і записує результат у потрібну колонку.

Основна функція:

```python
process_excel(file_path, task_description, inplace=True)
```

Підтримуються два режими:

- **online input**: Google Sheets / URL завантажується як локальний Excel у папку `data/`, після чого цей локальний файл заповнюється даними;
- **offline input**: локальний `.xlsx` відкривається і за замовчуванням заповнюється на місці.

У фінальних Excel-файлах не додаємо службові колонки. Логи, джерела, помилки та статуси залишаються у notebook як report.

In [1]:
%pip install -U openpyxl requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Крок 2. Імпорт бібліотек і налаштування

In [2]:
import json
import math
import re
import shutil
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from copy import copy
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Optional
from urllib.parse import urlparse

import requests
from openpyxl import load_workbook

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

USER_AGENT = "FinalExcelEnrichment/1.0 (student project; Wikidata and OpenStreetMap APIs)"
REQUEST_TIMEOUT = 25

print(f"Project directory: {PROJECT_DIR}")

Project directory: c:\Study\AI_agent_LLM


## Крок 3. Online та offline input

Нижче вказані реальні Google Sheets з умови. У notebook вони використовуються як online input: система завантажує їх у `data/`, а потім заповнює ці локальні Excel-файли.

Якщо замість URL передати локальний шлях до `.xlsx`, та сама функція обробить offline-файл без зміни коду.

In [3]:
CAPITALS_URL = "https://docs.google.com/spreadsheets/d/1FUmSLgIY9uJQcmlBf38RhD3k7wX34aDK/edit?usp=sharing&ouid=109311532605858499090&rtpof=true&sd=true"
MOUNTAINS_URL = "https://docs.google.com/spreadsheets/d/1_XIMfsjBoq481fWzDOMLFX8nTmznpR80/edit?usp=sharing&ouid=109311532605858499090&rtpof=true&sd=true"

CAPITALS_LOCAL_NAME = "capitals_Pylypenko.xlsx"
MOUNTAINS_LOCAL_NAME = "mountains_Pylypenko.xlsx"

print("Online inputs are configured.")

Online inputs are configured.


## Крок 4. Допоміжні функції для читання Excel і завантаження URL

Google Sheets URL перетворюємо на export endpoint, щоб отримати справжній `.xlsx`. Це дозволяє використовувати ті самі `process_excel(...)` виклики для online та offline input.

In [4]:
def is_url(value: str | Path) -> bool:
    parsed = urlparse(str(value))
    return parsed.scheme in {"http", "https"}


def google_sheets_export_url(url: str) -> str:
    match = re.search(r"/spreadsheets/d/([^/]+)", url)
    if match:
        sheet_id = match.group(1)
        return f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=xlsx"
    return url


def download_excel(url: str, destination: Path) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    export_url = google_sheets_export_url(url)
    response = requests.get(export_url, headers={"User-Agent": USER_AGENT}, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    destination.write_bytes(response.content)
    print(f"Downloaded online Excel to: {destination} ({destination.stat().st_size} bytes)")
    return destination


def resolve_excel_input(file_path: str | Path, local_file_name: str | None = None) -> tuple[Path, str]:
    if is_url(file_path):
        parsed = urlparse(str(file_path))
        fallback_name = re.sub(r"[^A-Za-z0-9_-]+", "_", parsed.path.strip("/"))[:80] or "online_input"
        destination_name = local_file_name or f"{fallback_name}.xlsx"
        destination = DATA_DIR / destination_name
        return download_excel(str(file_path), destination), "online"
    return Path(file_path), "offline"


def read_sheet_records(file_path: str | Path) -> tuple[list[str], list[dict[str, Any]]]:
    workbook = load_workbook(file_path)
    sheet = workbook.active
    headers = [cell.value for cell in sheet[1]]
    records = []
    for row in sheet.iter_rows(min_row=2, values_only=True):
        if all(value is None for value in row):
            continue
        records.append({headers[i]: row[i] if i < len(row) else None for i in range(len(headers))})
    return headers, records


def preview_excel(file_path: str | Path, limit: int = 7) -> None:
    headers, records = read_sheet_records(file_path)
    print(f"File: {file_path}")
    print(f"Rows: {len(records)}")
    print(f"Columns: {headers}")
    for record in records[:limit]:
        print(record)

def get_header_map(sheet) -> dict[str, int]:
    return {str(cell.value): cell.column for cell in sheet[1] if cell.value is not None}


## Крок 5. Структура логів

Логи не записуються в Excel. Вони потрібні тільки для report у notebook: який запит сформовано, яке джерело використано, яке значення знайдено, чи була помилка.

In [5]:
@dataclass
class ProcessingLog:
    row_number: int
    scenario: str
    query: str
    value: Optional[float | int]
    status: str
    source: str
    details: str = ""
    error: str = ""

def print_logs(logs: list[ProcessingLog], limit: Optional[int] = None) -> None:
    selected_logs = logs if limit is None else logs[:limit]
    for log in selected_logs:
        print("-" * 70)
        print(f"row={log.row_number} scenario={log.scenario} status={log.status}")
        print(f"query: {log.query}")
        print(f"value: {log.value}")
        print(f"source: {log.source}")
        if log.details:
            print(f"details: {log.details}")
        if log.error:
            print(f"error: {log.error}")

## Крок 6. Аналіз задачі та визначення сценарію

Функція не прив'язана до конкретного імені Excel-файлу. Вона визначає сценарій за колонками та текстом задачі.

In [6]:
def normalize_text(value: Any) -> str:
    return str(value or "").strip().lower()

def detect_target_column(headers: list[str], task_description: str, scenario: str) -> str:
    task = normalize_text(task_description)
    for header in headers:
        if normalize_text(header) in task:
            return header
    if scenario == "capital_distance" and "distance" in headers:
        return "distance"
    if scenario == "mountain_height" and "height" in headers:
        return "height"
    raise ValueError("Could not determine target column from task description and headers")

def analyze_task(headers: list[str], task_description: str) -> dict[str, str]:
    header_set = {normalize_text(header) for header in headers}
    task = normalize_text(task_description)

    has_capital_columns = {"capital_from", "country_from", "capital_to", "country_to"}.issubset(header_set)
    has_mountain_columns = {"mountain", "country"}.issubset(header_set)

    if has_capital_columns and ("distance" in header_set or "відстан" in task or "distance" in task):
        scenario = "capital_distance"
    elif has_mountain_columns and ("height" in header_set or "висот" in task or "height" in task):
        scenario = "mountain_height"
    else:
        raise ValueError(
            "Unsupported task. Expected either capital distance columns or mountain height columns."
        )

    target_column = detect_target_column(headers, task_description, scenario)
    return {"scenario": scenario, "target_column": target_column}


## Крок 7. Реальне добування координат столиць

Для координат використовуємо Nominatim/OpenStreetMap API. Це не ручний словник і не пам'ять LLM. Після отримання координат рахуємо пряму відстань через haversine по поверхні Землі.

In [7]:
class NominatimGeocoder:
    def __init__(self, min_delay_seconds: float = 1.0):
        self.cache: dict[tuple[str, str], dict[str, Any]] = {}
        self.min_delay_seconds = min_delay_seconds
        self.last_request_at = 0.0

    def geocode(self, capital: str, country: str) -> dict[str, Any]:
        key = (capital.strip().lower(), country.strip().lower())
        if key in self.cache:
            return self.cache[key]

        elapsed = time.time() - self.last_request_at
        if elapsed < self.min_delay_seconds:
            time.sleep(self.min_delay_seconds - elapsed)

        search_query = f"{capital}, {country}"
        response = requests.get(
            "https://nominatim.openstreetmap.org/search",
            params={"q": search_query, "format": "json", "limit": 1},
            headers={"User-Agent": USER_AGENT},
            timeout=REQUEST_TIMEOUT,
        )
        self.last_request_at = time.time()
        response.raise_for_status()
        data = response.json()
        if not data:
            raise ValueError(f"Coordinates not found for {search_query}")

        result = {
            "lat": float(data[0]["lat"]),
            "lon": float(data[0]["lon"]),
            "display_name": data[0].get("display_name", search_query),
            "source": "Nominatim/OpenStreetMap",
        }
        self.cache[key] = result
        return result

def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    earth_radius_km = 6371.0088
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)
    a = math.sin(delta_phi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return earth_radius_km * c

geocoder = NominatimGeocoder()

## Крок 8. Реальне добування висоти гір

Для висот використовуємо Wikidata API. Система шукає entity за назвою гори, перевіряє наявність властивості `P2044` elevation above sea level і бере значення в метрах.

In [8]:
class WikidataMountainHeightClient:
    METRE_UNIT = "http://www.wikidata.org/entity/Q11573"

    def __init__(self):
        self.cache: dict[str, dict[str, Any]] = {}

    def search_entities(self, mountain: str) -> list[dict[str, Any]]:
        response = requests.get(
            "https://www.wikidata.org/w/api.php",
            params={
                "action": "wbsearchentities",
                "search": mountain,
                "language": "en",
                "format": "json",
                "limit": 7,
            },
            headers={"User-Agent": USER_AGENT},
            timeout=REQUEST_TIMEOUT,
        )
        response.raise_for_status()
        return response.json().get("search", [])

    def get_entity(self, qid: str) -> dict[str, Any]:
        response = requests.get(
            f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.json",
            headers={"User-Agent": USER_AGENT},
            timeout=REQUEST_TIMEOUT,
        )
        response.raise_for_status()
        return response.json()["entities"][qid]

    def extract_height_m(self, entity: dict[str, Any]) -> Optional[float]:
        claims = entity.get("claims", {})
        if "P2044" not in claims:
            return None

        preferred_values = []
        normal_values = []
        for claim in claims["P2044"]:
            value = claim.get("mainsnak", {}).get("datavalue", {}).get("value", {})
            amount = value.get("amount")
            unit = value.get("unit")
            if amount is None or unit != self.METRE_UNIT:
                continue
            numeric_value = float(amount)
            if claim.get("rank") == "preferred":
                preferred_values.append(numeric_value)
            elif claim.get("rank") != "deprecated":
                normal_values.append(numeric_value)

        if preferred_values:
            return preferred_values[0]
        if normal_values:
            return normal_values[0]
        return None

    def get_height(self, mountain: str, country: str = "") -> dict[str, Any]:
        key = mountain.strip().lower()
        if key in self.cache:
            return self.cache[key]

        candidates = self.search_entities(mountain)
        errors = []
        for candidate in candidates:
            qid = candidate["id"]
            description = normalize_text(candidate.get("description", ""))
            label = candidate.get("label", qid)
            if candidates.index(candidate) > 0 and "mountain" not in description and "peak" not in description:
                continue
            try:
                entity = self.get_entity(qid)
                height = self.extract_height_m(entity)
                if height is not None:
                    result = {
                        "height_m": round(height),
                        "source": f"Wikidata {qid}",
                        "details": f"{label}: {candidate.get('description', '')}",
                    }
                    self.cache[key] = result
                    return result
            except Exception as exc:
                errors.append(f"{qid}: {exc}")
        raise ValueError(f"Height not found for {mountain}. Errors: {'; '.join(errors[:3])}")

height_client = WikidataMountainHeightClient()

## Крок 9. Multi-agent архітектура

Система оформлена як набір спеціалізованих агентів:

- `InputAgent` приймає online/offline Excel і готує локальний файл для обробки.
- `TaskAnalyzerAgent` аналізує колонки та текст задачі.
- `ResearchAgent` добуває дані з реальних джерел: Nominatim/OpenStreetMap для координат і Wikidata для висот.
- `WriterAgent` записує значення тільки в потрібну колонку Excel.
- `ValidationAgent` перевіряє фінальний файл і відсутність службових колонок.
- `ExcelEnrichmentOrchestrator` керує повним циклом.

Це не ручний словник: дані реально добуваються через API, а для столиць пряма відстань рахується через haversine.

In [9]:
class InputAgent:
    """Приймає online або offline Excel і повертає локальний шлях для обробки."""

    def prepare(self, file_path: str | Path, local_file_name: str | None = None) -> tuple[Path, str]:
        return resolve_excel_input(file_path, local_file_name=local_file_name)


class TaskAnalyzerAgent:
    """Визначає сценарій і цільову колонку за структурою Excel та описом задачі."""

    def analyze(self, sheet, task_description: str) -> dict[str, str]:
        header_map = get_header_map(sheet)
        headers = list(header_map.keys())
        analysis = analyze_task(headers, task_description)
        return {**analysis, "headers": headers}


class ResearchAgent:
    """Добуває потрібну інформацію з реальних API-джерел."""

    def research_capital_distance(self, row_idx: int, sheet, header_map: dict[str, int]) -> tuple[Optional[int], ProcessingLog]:
        capital_from = sheet.cell(row_idx, header_map["Capital_From"]).value
        country_from = sheet.cell(row_idx, header_map["Country_From"]).value
        capital_to = sheet.cell(row_idx, header_map["Capital_To"]).value
        country_to = sheet.cell(row_idx, header_map["Country_To"]).value
        query = f"coordinates of {capital_from}, {country_from}; coordinates of {capital_to}, {country_to}"

        try:
            from_coords = geocoder.geocode(str(capital_from), str(country_from))
            to_coords = geocoder.geocode(str(capital_to), str(country_to))
            distance = round(haversine_km(from_coords["lat"], from_coords["lon"], to_coords["lat"], to_coords["lon"]))
            log = ProcessingLog(
                row_number=row_idx,
                scenario="capital_distance",
                query=query,
                value=distance,
                status="success",
                source="Nominatim/OpenStreetMap + haversine",
                details=f"{from_coords['display_name']} -> {to_coords['display_name']}",
            )
            return distance, log
        except Exception as exc:
            log = ProcessingLog(
                row_number=row_idx,
                scenario="capital_distance",
                query=query,
                value=None,
                status="failed",
                source="Nominatim/OpenStreetMap + haversine",
                error=str(exc),
            )
            return None, log

    def research_mountain_height(self, row_idx: int, mountain: str, country: str) -> tuple[int | None, ProcessingLog]:
        query = f"Wikidata elevation above sea level for {mountain}, {country}"
        try:
            result = height_client.get_height(mountain, country)
            height = result["height_m"]
            log = ProcessingLog(
                row_number=row_idx,
                scenario="mountain_height",
                query=query,
                value=height,
                status="success",
                source=result["source"],
                details=result["details"],
            )
            return height, log
        except Exception as exc:
            log = ProcessingLog(
                row_number=row_idx,
                scenario="mountain_height",
                query=query,
                value=None,
                status="failed",
                source="Wikidata API",
                error=str(exc),
            )
            return None, log


class WriterAgent:
    """Записує знайдені значення в потрібну колонку без додавання службових колонок."""

    def write_value(self, sheet, row_idx: int, target_column_index: int, value: int | float | None) -> None:
        if value is not None:
            sheet.cell(row_idx, target_column_index).value = value


class ValidationAgent:
    """Перевіряє фінальний Excel-файл."""

    service_columns = {"status", "source", "confidence", "error", "details"}

    def validate(self, file_path: str | Path, target_column: str) -> None:
        headers, records = read_sheet_records(file_path)
        filled = sum(record.get(target_column) not in (None, "") for record in records)
        print("=" * 70)
        print(f"Validated: {file_path}")
        print(f"Rows: {len(records)}")
        print(f"Columns: {headers}")
        print(f"Filled {target_column}: {filled}/{len(records)}")
        assert target_column in headers, f"Missing target column: {target_column}"
        assert not (set(map(normalize_text, headers)) & self.service_columns), "Service columns should not be in final Excel"
        assert filled == len(records), f"Not all values are filled in {target_column}"
        for record in records[:10]:
            print(record)


class ExcelEnrichmentOrchestrator:
    """Керує повним multi-agent циклом збагачення Excel."""

    def __init__(self):
        self.input_agent = InputAgent()
        self.task_agent = TaskAnalyzerAgent()
        self.research_agent = ResearchAgent()
        self.writer_agent = WriterAgent()
        self.validation_agent = ValidationAgent()

    def process(
        self,
        file_path: str | Path,
        task_description: str,
        *,
        inplace: bool = True,
        output_path: str | Path | None = None,
        local_file_name: str | None = None,
    ) -> dict[str, Any]:
        local_path, input_mode = self.input_agent.prepare(file_path, local_file_name=local_file_name)
        if not local_path.exists():
            raise FileNotFoundError(local_path)

        work_path = local_path
        if not inplace:
            work_path = Path(output_path) if output_path else default_enriched_path(local_path)
            work_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(local_path, work_path)

        workbook = load_workbook(work_path)
        sheet = workbook.active
        header_map = get_header_map(sheet)
        analysis = self.task_agent.analyze(sheet, task_description)
        scenario = analysis["scenario"]
        target_column = analysis["target_column"]

        print("=" * 70)
        print(f"Input mode: {input_mode}")
        print(f"Processing Excel: {work_path}")
        print(f"Task: {task_description}")
        print(f"Scenario: {scenario}")
        print(f"Target column: {target_column}")
        print("=" * 70)

        logs = []
        target_column_index = header_map[target_column]

        if scenario == "capital_distance":
            for row_idx in range(2, sheet.max_row + 1):
                value, log = self.research_agent.research_capital_distance(row_idx, sheet, header_map)
                self.writer_agent.write_value(sheet, row_idx, target_column_index, value)
                logs.append(log)
        elif scenario == "mountain_height":
            rows = []
            for row_idx in range(2, sheet.max_row + 1):
                mountain = str(sheet.cell(row_idx, header_map["Mountain"]).value)
                country = str(sheet.cell(row_idx, header_map["Country"]).value)
                rows.append((row_idx, mountain, country))

            logs_by_row = {}
            with ThreadPoolExecutor(max_workers=5) as executor:
                future_map = {
                    executor.submit(self.research_agent.research_mountain_height, row_idx, mountain, country): (row_idx, mountain, country)
                    for row_idx, mountain, country in rows
                }
                for future in as_completed(future_map):
                    row_idx, _, _ = future_map[future]
                    value, log = future.result()
                    self.writer_agent.write_value(sheet, row_idx, target_column_index, value)
                    logs_by_row[row_idx] = log
            logs = [logs_by_row[row_idx] for row_idx, _, _ in rows]
        else:
            raise ValueError(f"Unsupported scenario: {scenario}")

        workbook.save(work_path)
        success_count = sum(log.status == "success" for log in logs)
        print(f"Saved Excel file: {work_path}")
        print(f"Success: {success_count}/{len(logs)}")

        return {
            "input_mode": input_mode,
            "input_file": str(local_path),
            "output_file": str(work_path),
            "scenario": scenario,
            "target_column": target_column,
            "logs": logs,
        }


orchestrator = ExcelEnrichmentOrchestrator()


def process_excel(
    file_path: str | Path,
    task_description: str,
    *,
    inplace: bool = True,
    output_path: str | Path | None = None,
    local_file_name: str | None = None,
) -> dict[str, Any]:
    return orchestrator.process(
        file_path=file_path,
        task_description=task_description,
        inplace=inplace,
        output_path=output_path,
        local_file_name=local_file_name,
    )

## Крок 10. Запуск для online capitals.xlsx

Передаємо Google Sheets URL. `InputAgent` завантажує реальний Excel у `data/capitals_Pylypenko.xlsx`, `ResearchAgent` добуває координати столиць через Nominatim/OpenStreetMap, а `WriterAgent` заповнює колонку `distance` у цьому локальному файлі.

In [10]:
capitals_result = process_excel(
    file_path=CAPITALS_URL,
    task_description="знайди пряму відстань між столицями в км для колонки distance",
    inplace=True,
    local_file_name=CAPITALS_LOCAL_NAME,
)
print_logs(capitals_result["logs"])

Downloaded online Excel to: c:\Study\AI_agent_LLM\data\capitals_Pylypenko.xlsx (9019 bytes)
Input mode: online
Processing Excel: c:\Study\AI_agent_LLM\data\capitals_Pylypenko.xlsx
Task: знайди пряму відстань між столицями в км для колонки distance
Scenario: capital_distance
Target column: distance
Saved Excel file: c:\Study\AI_agent_LLM\data\capitals_Pylypenko.xlsx
Success: 10/10
----------------------------------------------------------------------
row=2 scenario=capital_distance status=success
query: coordinates of Kyiv, Ukraine; coordinates of Paris, France
value: 2024
source: Nominatim/OpenStreetMap + haversine
details: Київ, Україна -> Paris, Île-de-France, France métropolitaine, France
----------------------------------------------------------------------
row=3 scenario=capital_distance status=success
query: coordinates of London, United Kingdom; coordinates of Rome, Italy
value: 1434
source: Nominatim/OpenStreetMap + haversine
details: Greater London, England, United Kingdom -> 

## Крок 11. Запуск для online mountains.xlsx

Передаємо Google Sheets URL. `InputAgent` завантажує реальний Excel у `data/mountains_Pylypenko.xlsx`, `ResearchAgent` добуває висоту гір через Wikidata, а `WriterAgent` заповнює колонку `height` у цьому локальному файлі.

In [11]:
mountains_result = process_excel(
    file_path=MOUNTAINS_URL,
    task_description="додай висоту гір у метрах до колонки height",
    inplace=True,
    local_file_name=MOUNTAINS_LOCAL_NAME,
)
print_logs(mountains_result["logs"])

Downloaded online Excel to: c:\Study\AI_agent_LLM\data\mountains_Pylypenko.xlsx (8801 bytes)
Input mode: online
Processing Excel: c:\Study\AI_agent_LLM\data\mountains_Pylypenko.xlsx
Task: додай висоту гір у метрах до колонки height
Scenario: mountain_height
Target column: height
Saved Excel file: c:\Study\AI_agent_LLM\data\mountains_Pylypenko.xlsx
Success: 10/10
----------------------------------------------------------------------
row=2 scenario=mountain_height status=success
query: Wikidata elevation above sea level for Mount Everest, Nepal/China
value: 8849
source: Wikidata Q513
details: Mount Everest: Earth's highest mountain above sea level, located in the Mahalangur Himal sub-range of the Himalayas
----------------------------------------------------------------------
row=3 scenario=mountain_height status=success
query: Wikidata elevation above sea level for K2, Pakistan/China
value: 8611
source: Wikidata Q43512
details: K2: 2nd-highest mountain on Earth
-------------------------

## Крок 12. Фінальна перевірка заповнених Excel-файлів

`ValidationAgent` перевіряє локальні файли в `data/`: усі рядки на місці, потрібні колонки заповнені, службові колонки не додані.

In [12]:
orchestrator.validation_agent.validate(capitals_result["output_file"], "distance")
orchestrator.validation_agent.validate(mountains_result["output_file"], "height")

Validated: c:\Study\AI_agent_LLM\data\capitals_Pylypenko.xlsx
Rows: 10
Columns: ['Capital_From', 'Country_From', 'Capital_To', 'Country_To', 'distance']
Filled distance: 10/10
{'Capital_From': 'Kyiv', 'Country_From': 'Ukraine', 'Capital_To': 'Paris', 'Country_To': 'France', 'distance': 2024}
{'Capital_From': 'London', 'Country_From': 'United Kingdom', 'Capital_To': 'Rome', 'Country_To': 'Italy', 'distance': 1434}
{'Capital_From': 'Paris', 'Country_From': 'France', 'Capital_To': 'Berlin', 'Country_To': 'Germany', 'distance': 877}
{'Capital_From': 'Berlin', 'Country_From': 'Germany', 'Capital_To': 'Vienna', 'Country_To': 'Austria', 'distance': 524}
{'Capital_From': 'Warsaw', 'Country_From': 'Poland', 'Capital_To': 'Kyiv', 'Country_To': 'Ukraine', 'distance': 685}
{'Capital_From': 'Rome', 'Country_From': 'Italy', 'Capital_To': 'Madrid', 'Country_To': 'Spain', 'distance': 1363}
{'Capital_From': 'Madrid', 'Country_From': 'Spain', 'Capital_To': 'Lisbon', 'Country_To': 'Portugal', 'distance':

## Висновок

У проєкті реалізовано multi-agent AI-систему з універсальним інтерфейсом:

```python
process_excel(file_path, task_description, inplace=True)
```

Система підтримує:

1. **Online input** — Google Sheets або інше URL-посилання на Excel.
2. **Offline input** — локальний `.xlsx` файл.
3. **Автоматичний аналіз задачі** — визначення сценарію та цільової колонки за структурою таблиці й описом користувача.
4. **Реальне добування даних** — Nominatim/OpenStreetMap для координат столиць і Wikidata для висот гір.
5. **Коректну формулу відстані** — haversine для прямої відстані по поверхні Землі.
6. **Чистий Excel-результат** — заповнюється тільки потрібна колонка, без службових колонок.
7. **Report у notebook** — логи показують запити, джерела, статуси та знайдені значення.

Фінальні файли для здачі після запуску notebook:

- `data/capitals_Pylypenko.xlsx`
- `data/mountains_Pylypenko.xlsx`